# אילו הונאות מטרידות משתמשים? — ניסוי ראשון

מנתחים טקסטים **באנגלית** ומציגים סיכום בעברית באמצעות Hugging Face.
זהו אבטיפוס לניתוח, עדיין ללא איסוף מ־Reddit או ריצה מתוזמנת.

כל הטקסטים להלן **מומצאים לצורך הדגמה**, ואינם ממצאים על Reddit.
בחר את הקרנל של הפרויקט והריץ לפי הסדר. אין צורך במפתח API.
הרצת תא טעינת המודל מורידה מודל חדש (מאות MB) למטמון המקומי; הניתוח עצמו מקומי.

השלבים: נתונים ← ניקוי כפילויות ← סיווג הונאה וחשש ← דוח ← בדיקה ידנית.

## 1. סביבת העבודה
הניסוי משתמש בספריות מהמחברת הקודמת ובתוסף `sentencepiece`.
אם חסרות ספריות, הסר את `#` משורת ההתקנה והריץ אותה בתוך המחברת.

In [ ]:
# %pip install -r requirements-reddit-scam.txt
import os
import sys
from pathlib import Path
from collections import Counter
from IPython.display import Markdown, display

os.environ["HF_HOME"] = str(Path.cwd() / ".hf-cache")
print(sys.executable)

## 2. נתוני הדגמה בלבד
יחידת הספירה כרגע היא טקסט ייחודי, לא אדם ולא קורבן מאומת.
יש גם טקסט לא רלוונטי כדי לבדוק שהמודל מסוגל להימנע מסיווג להונאה.
לא הוספנו קישורי Reddit מומצאים.

In [ ]:
DATA_MODE = "synthetic"
posts = [
    {"id": "demo-1", "text": "A text says my bank account is locked and asks me to enter my password on a website. I cannot tell if this message is real.", "url": None},
    {"id": "demo-2", "text": "Someone pretending to be my bank called and asked for my login code. I am scared they will steal my savings.", "url": None},
    {"id": "demo-3", "text": "An online recruiter wants an upfront payment before I can start a remote job. How can I check whether the job is legitimate?", "url": None},
    {"id": "demo-4", "text": "My online boyfriend convinced me to transfer money and then disappeared. I need help recovering the money.", "url": None},
    {"id": "demo-5", "text": "A crypto investment website will not let me withdraw my money unless I pay another fee. How can I get my funds back?", "url": None},
    {"id": "demo-6", "text": "I am making pasta for dinner and looking for a tomato sauce recipe.", "url": None},
]

def deduplicate(records):
    seen_ids, seen_texts, result = set(), set(), []
    for record in records:
        normalized = " ".join(record["text"].split()).casefold()
        if not normalized or normalized in {"[deleted]", "[removed]"}:
            continue
        if record["id"] in seen_ids or normalized in seen_texts:
            continue
        seen_ids.add(record["id"])
        seen_texts.add(normalized)
        result.append(record.copy())
    return result

unique_posts = deduplicate(posts)
print(f"Demo mode: {DATA_MODE}; {len(unique_posts)} unique texts")

## 3. הגדרת הקטגוריות
Zero-shot classification משווה טקסט לתיאורים שנגדיר, ללא אימון נוסף.
הקטגוריות הן השערות שלנו וצריך לשפר אותן לפי נתונים אמיתיים.

נפריד בין סוג הונאה לחשש. בגרסה זו נבחר סוג אחד וחשש אחד לכל טקסט;
במציאות עשויים להיות כמה. `other` מאפשר לסמן הונאה שלא כיסינו.

In [ ]:
SCAM_LABELS = {
    "bank impersonation or phishing for passwords and login codes": "התחזות לבנק ודיוג פרטי גישה",
    "fake jobs and recruitment scams": "הונאות עבודה וגיוס",
    "romance scams involving an online relationship": "הונאות רומנטיות",
    "fraudulent investments and cryptocurrency scams": "הונאות השקעה וקריפטו",
    "online shopping and marketplace scams": "הונאות קנייה ומכירה",
    "fake technical support scams": "הונאות תמיכה טכנית",
    "another type of online scam": "הונאה אחרת — לבדיקה",
    "a topic unrelated to online scams": "לא רלוונטי",
}
CONCERN_LABELS = {
    "difficulty checking whether a message, person or offer is legitimate": "קושי לבדוק אמינות",
    "fear of losing money": "חשש מאובדן כסף",
    "difficulty recovering money after being scammed": "קושי בהחזרת כסף",
    "fear of identity theft or unauthorized account access": "חשש מגניבת זהות או חשבון",
    "emotional distress, shame or loss of trust": "מצוקה, בושה ואובדן אמון",
    "another concern or no clearly expressed concern": "חשש אחר או לא מפורש",
}
# ספים התחלתיים בלבד, לא ספים מכוילים ולא הסתברות לנכונות.
MIN_SCORE = 0.55
MIN_MARGIN = 0.15
MODEL_ID = "MoritzLaurer/deberta-v3-base-zeroshot-v2.0"


## 4. טעינת המודל
התא הבא מוריד את המודל בהרצה הראשונה. הוא מסווג טקסט, ואינו מודל שיחה.
המודל מוגבל לאורך קלט; טקסטים ארוכים ייחתכו ויסומנו לבדיקה ידנית.
[מקור והוראות שימוש במודל](https://huggingface.co/MoritzLaurer/deberta-v3-base-zeroshot-v2.0)

In [ ]:
from transformers import pipeline

classifier = pipeline("zero-shot-classification", model=MODEL_ID, device="cpu")
print("Model loaded:", MODEL_ID)

## 5. ניתוח
נשמור גם את הצעת המודל כשהיא לא מספיק ברורה, אך לא נספור אותה כממצא מאושר.
כל פלט דורש בדיקה אנושית; ציוני המודל אינם מודדים את עוצמת הפחד.
הסיווג מוגבל לטקסט הנתון ואינו מאמת שהונאה אכן התרחשה.

In [ ]:
def classify_axis(text, labels, scorer):
    output = scorer(
        text,
        candidate_labels=list(labels),
        hypothesis_template="This text is about {}.",
        multi_label=False,
    )
    score, runner_up = map(float, output["scores"][:2])
    accepted = score >= MIN_SCORE and score - runner_up >= MIN_MARGIN
    return {
        "label": labels[output["labels"][0]] if accepted else "לא ברור — לבדיקה",
        "suggested": labels[output["labels"][0]],
        "score": score,
        "margin": score - runner_up,
    }

def analyze(records, scorer):
    results = []
    for record in deduplicate(records):
        # היפותזת הסיווג צורכת גם היא חלק מחלון ההקשר של המודל.
        token_count = len(scorer.tokenizer.encode(record["text"], add_special_tokens=False))
        scam = classify_axis(record["text"], SCAM_LABELS, scorer)
        concern = None
        if scam["label"] != "לא רלוונטי":
            concern = classify_axis(record["text"], CONCERN_LABELS, scorer)
        results.append({**record, "scam": scam, "concern": concern,
                        "long_text_review": token_count > 400})
    return results

results = analyze(unique_posts, classifier)
for row in results:
    print(row["id"], "|", row["scam"]["label"], "|",
          row["concern"]["label"] if row["concern"] else "—",
          "| long text:", row["long_text_review"])


## 6. דוח בעברית
הדירוג הוא לפי מספר טקסטים שסווגו בביטחון מספק במדגם, ולא לפי נזק או מספר משתמשים.
טקסטים ארוכים מסומנים לבדיקה ואינם נכללים בדירוג.
אפשר לפתוח מחדש כל טקסט מהפלט המפורט ולבדוק את הסיווג.

In [ ]:
def make_report(rows, mode):
    heading = "הדגמה סינתטית בלבד — אין כאן ממצאים מ־Reddit" if mode == "synthetic" else "תוצאות מדגם — טעונות בדיקה אנושית"
    excluded = {"לא רלוונטי", "לא ברור — לבדיקה", "הונאה אחרת — לבדיקה"}
    ranked = [r for r in rows if r["scam"]["label"] not in excluded and not r["long_text_review"]]
    counts = Counter(r["scam"]["label"] for r in ranked)
    lines = [f"# {heading}", "", f"נבדקו {len(rows)} טקסטים ייחודיים; {len(ranked)} נכללו בדירוג.", "",
             "| סוג הונאה | מספר טקסטים |", "|---|---:|"]
    lines += [f"| {label} | {count} |" for label, count in counts.most_common()]
    lines += ["", "### חששות בטקסטים שנכללו בדירוג", ""]
    concerns = Counter(r["concern"]["label"] for r in ranked if r["concern"])
    lines += [f"- {label}: {count}" for label, count in concerns.most_common()]
    lines += ["", "### מקורות לבדיקה", ""]
    for row in rows:
        lines.append(f"- {row['id']}: {row['url'] or 'טקסט מומצא, ללא קישור מקור'}")
    lines += ["", "השכיחות מתייחסת למדגם בלבד. קטגוריות וספים עדיין לא כוילו; אין להסיק ממנה את שכיחות ההונאות בעולם."]
    return "\n".join(lines)

display(Markdown(make_report(results, DATA_MODE)))

## 7. איך נתקדם ל־Reddit אמיתי?
1. נגדיר את השימוש ואת היקף האיסוף: אנגלית, פוסטים ציבוריים מ־r/Scams, פיילוט קטן ומוגדר בזמן.
2. נבקש גישה דרך [מסלול המפתחים הרשמי](https://support.reddithelp.com/hc/en-us/articles/14945211791892-Developer-Platform-Accessing-Reddit-Data).
   Reddit מפנה מפתחים קודם ל־Devvit; אינטגרציה חיצונית דורשת את מסלול האישור המתאים.
   יש לתאר במפורש סיווג AI מקומי ואת אופי השימוש בפועל (לימודי או מסחרי).
3. לאחר אישור נבנה אספן בהתאם לממשק ולמגבלות שאושרו. נשמור קישור, מזהה ותאריך, בלי צורך בשמות משתמשים.
4. נתייג ידנית מדגם ונבדוק רלוונטיות, סוג הונאה והחשש, כולל טעויות ושלילות. נכייל ספים וקטגוריות לפני הסקת מסקנות.
5. רק לאחר מכן נוסיף תגובות, השוואת חלונות זמן ותזמון. תגובות לא ייספרו כדיונים עצמאיים.

כרגע אין איסוף חי, אימות נזקים, חילוץ סכומים, מדידת מגמות או סוכן אוטונומי.
נתוני Reddit לא ישמשו לאימון מודל במסגרת הניסוי.
[מדיניות הגישה](https://support.reddithelp.com/hc/en-us/articles/16471395473812-Moderation-Bots-Tooling)